### All Imports

In [583]:
import os
from langgraph.graph import StateGraph
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict, List, Literal
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from tavily import TavilyClient
from pydantic import BaseModel, PositiveInt
from pprint import pprint
from IPython.display import Markdown

In [567]:
load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [568]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)

In [569]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [570]:
# Prompts - Read Content from Files
f = open("./prompts/generations.md")
GENERATION_PROMPT = f.read()
f = open("./prompts/optimisation.md")
OPTIMISATION_PROMPT = f.read()
f = open("./prompts/scoring.md")
SCORING_PROMPT = f.read()
f = open("./prompts/validation.md")
VALIDATION_PROMPT = f.read()

In [571]:
# Output of scroing agent
class Score(BaseModel):
    score: PositiveInt
    description: str

# Output of generation offre
class Offre(BaseModel):
    offre: str

# Output of validation offre
class IsValid(BaseModel):
    is_valid: bool
    
class MarketingState(TypedDict):
    user_input: str
    client_data: str
    score: Score
    offer: Offre
    valid: IsValid
    optimized_version: Offre
    status: Literal["INIT", "FETCHING", "SCORING", "GENERATION", "VALIDATION", "OPTMISATION", "END"]

#### Fetcher Agent

In [572]:
# Fetcher Agent
"""
    In Backend We Should Handle This Part
        --> User Enter Prompt
        --> User Select CSV, Excel
        --> Use Use MCP 
"""

# This agent can use tools to connect to DB, Read execel files, ...
def fetcher_agent(state: MarketingState):
    """Get Client Data Just by Name"""
    client_data = {
        "name": "ahmed",
        "customer_id": "CUST12345",
        "email": f"ahmed@example.com",
        "phone": "+33 1 23 45 67 89",
        "address": {
            "street": "123 Rue de Exemple",
            "city": "Paris",
            "postal_code": "75001",
            "country": "France",
        },
        "achat_history": [
            {"date": "2026-07-20", "product": "Parfum", "amount": 59.90, "quantity": 1},
            {"date": "2026-06-15", "product": "Chaussures", "amount": 120.00, "quantity": 1},
            {"date": "2026-05-02", "product": "T-shirt", "amount": 29.90, "quantity": 2},
        ],
        "total_spent": 209.70,
        "status": "active",
    }
    return {"client_data": str(client_data), "status": "SCORING"}


#### Scoring Agent

In [573]:
# Tools for Scoring Agent
@tool("getHowToScoreUser")
def getHowToScoreUser(query: str, limit: int) -> list:
    """
        Search On Internet Using Of How to Score Custmor in Marekting
        
        Args:
            query: Search term to look for
            limit: Maximum number of results to return
    """
    print("TOOL CALL: getHowToScoreUser")
    response = tavily_client.search(query=query, max_results=limit)
    results = response.get("results", [])
    content = []
    for res in results:
        content.append({"url": res.get("url", ""), "content": res.get("content", "")})
    return content


# Scoring Agent
def scoring_agent(state: MarketingState):
    print("Scoring ...")
    messages = [
        SystemMessage(content=SCORING_PROMPT),
        HumanMessage(content=f"Here is Customer Data: {state.get("client_data", "NONE")} "),
    ]
    # bind tools give choose to llm
    basic_llm_scoring = basic_llm.bind_tools([getHowToScoreUser])
    response = basic_llm_scoring.with_structured_output(Score).invoke(messages)
    print(response)
    return {"score": response, "status": "GENERATION"}


In [574]:
# Test Agent
client_data = fetcher_agent({})["client_data"]
state: MarketingState = {
    "user_input": "Please score this user",
    "client_data": client_data,
    "marketing_offer": "",
    "score": None,
    "status": "SCORING",
}
result = scoring_agent(state)

Scoring ...
score=85 description='The client, Ahmed, has a score of 85 due to his active status and a moderate level of engagement with the company. He has made three purchases, with a total spent of 209.7, indicating a certain level of loyalty. However, the frequency of his purchases is relatively low, and the average amount spent per transaction is moderate. This suggests that Ahmed is a reliable but not highly active customer. His address and contact information are complete, which is a positive aspect. Nevertheless, there is room for improvement in terms of increasing the frequency and average value of his purchases to reach a higher score.'


In [575]:
pprint(list(result.values())[0].score)
display(Markdown(list(result.values())[0].description))

85


The client, Ahmed, has a score of 85 due to his active status and a moderate level of engagement with the company. He has made three purchases, with a total spent of 209.7, indicating a certain level of loyalty. However, the frequency of his purchases is relatively low, and the average amount spent per transaction is moderate. This suggests that Ahmed is a reliable but not highly active customer. His address and contact information are complete, which is a positive aspect. Nevertheless, there is room for improvement in terms of increasing the frequency and average value of his purchases to reach a higher score.

#### Generation Agent

In [576]:
# Tools
@tool
def getContent():
    """
        Tool for getting Neccessary Information To Generate Marketing Offer Such as:
        - Busniss Type
        - Product Service
        - Target Audience - Place Audiance.
        - Compaign Duration
        - ...
    """
    return "None Information Here"
    

# Generation Agent
def generation_agent(state: MarketingState):
    # Customer Data or Customers Data + Business Info
    customer_data = state.get("client_data", "")
    messages = [SystemMessage(content=GENERATION_PROMPT), HumanMessage(content=f"{state.get("user_input", "generate Marketing Offre Based on Customer Data")} and here is customer data {state.get("client_data", "No Client Data")}")]
    
    basic_llm_generator = basic_llm.bind_tools([getContent])
    response = basic_llm_generator.with_structured_output(Offre).invoke(messages)
    return {"offre": response, "status": "VALIDATION"}


In [577]:
# Test Generation Agent
client_data = fetcher_agent({})["client_data"]
state: MarketingState = {
    "user_input": "",
    "client_data": client_data,
    "marketing_offer": "",
    "score": 85,
    "status": "SCORING",
}
result = generation_agent(state)

In [578]:
display(Markdown(result.get("offre").offre))

Bonjour Ahmed, 

Nous sommes ravis de vous proposer une offre spéciale en votre nom ! 

**Vous avez déjà dépensé 209,70€ chez nous**

Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !

**Découvrez nos meilleures offres**

* **Parfum exclusif** : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
* **Chaussures de luxe** : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
* **T-shirt premium** : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !

**N'oubliez pas de profiter de nos offres spéciales**

* **Livraison gratuite** : sur tous les commandes supérieures à 100€
* **Retour gratuit** : sur tous les produits achetés chez nous

**Cliquez ici pour découvrir nos offres** [lien vers la page d'offres]

Nous sommes impatients de vous voir chez nous !

Cordialement, 
L'équipe de [votre nom de l'entreprise]

#### Validation Agent

In [579]:
# Tools
@tool
def getValidationRequirements():
    """
        Get Rules To Validate The Offre
    """
    
    
    # Read File From Data Called validation.md
    return "no rules required yet"

# Validation Agent
def validation_agent(state: MarketingState):
    # Get Offre
    offre = state.get("offre", "no offre")
    print(offre)
    
    messages = [SystemMessage(content=VALIDATION_PROMPT), HumanMessage(content=f"Is that this offre valid {offre}")]
    basic_llm_generator = basic_llm.bind_tools([getValidationRequirements])
    response = basic_llm_generator.with_structured_output(IsValid).invoke(messages)
    print(type(response.is_valid))
    if response.is_valid:
        return {"offre": response, "status": "END"}
    return {"offre": response, "status": "OPTMISATION"}

In [580]:
# Test Validation Agent
offre = """Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]
"""

# Test Generation Agent
client_data = fetcher_agent({})["client_data"]
state: MarketingState = {
    "user_input": "",
    "client_data": client_data,
    "marketing_offer": "",
    "score": 85,
    "offre": offre,
    "status": "SCORING",
}
result = validation_agent(state)
print(result)

Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équip

#### Optimisation Offre

In [581]:
# Tools for Optimisation offre


# Agent for offre Optmisation
def optmisation_agent(state: MarketingState):
    # Get Offre
    offre = state.get("offre", "None")
    messages = [SystemMessage(content=OPTIMISATION_PROMPT), HumanMessage(content=f"Optimise this Offre: {offre}")]
    
    response = basic_llm.with_structured_output(Offre).invoke(messages)
    return {"offre": response, "status": "END"}

In [582]:
# Test Validation Agent
offre = """Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]
"""

# Test Generation Agent
client_data = fetcher_agent({})["client_data"]
state: MarketingState = {
    "user_input": "",
    "client_data": client_data,
    "marketing_offer": "",
    "score": 85,
    "offre": offre,
    "status": "SCORING",
}
result = optmisation_agent(state)
display(Markdown(result.get('offre').offre))

Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]

#### Build graph

In [585]:
# Routing
def router(state: MarketingState):
    return state["status"]

In [584]:
workflow = StateGraph(MarketingState)

In [ ]:
workflow.add_node("fetcher_agent", fetcher_agent)
workflow.add_node("scoring_agent", scoring_agent)
workflow.add_node("generation_agent", generation_agent)
workflow.add_node("validation_agent", validation_agent)
workflow.add_node("optmisation_agent", optmisation_agent)

workflow.set_entry_point(fetcher_agent)


